In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

In [3]:
# Đọc file phân loại để lấy danh sách mã hàng Nhóm 1
classification_df = pd.read_excel('fsku_classification_results.xlsx')
nhom1_fskus = classification_df[classification_df['Phân Nhóm'] == 'Nhom 1: >= 12 thang']['fsku'].unique()

In [7]:
print(nhom1_fskus)

['00.KCB.100.NS' '00.KCB.150.NS' '00.KCB.200.NS' ... '41.L1.6060.U6155'
 '43.HC.ALSN3.TRONG' '43.HC.ALSN5.TRANG']


In [17]:
# Đọc dữ liệu bán hàng chi tiết
df = pd.read_excel('Dữ liệu bán hàng từ 2023 đến 09.2025 -new.xlsx')


In [12]:
print(df)

                          fsku year_month   qty
0      45.TB.LUX.CRC.82.45.2.K    2025-06   1.0
1      45.TB.LUX.CRC.82.45.2.K    2025-07   6.0
2      45.TB.LUX.CRC.82.45.2.K    2025-02   1.0
3      45.TB.LUX.CRC.82.45.2.K    2025-08   2.0
4      45.PK.LUX.TS.120.120.62    2025-09   9.0
...                        ...        ...   ...
28124     31.L1.LEO-100100-008    2025-05  14.0
28125       20.L1.100100.91123    2025-04   2.0
28126     31.L1.LEO-100100-009    2023-06   1.0
28127     31.L1.LEO-100100-004    2025-06   4.0
28128     31.L1.LEO-100100-009    2025-03   5.0

[28129 rows x 3 columns]


In [18]:
# Chuyển đổi cột thời gian và sản lượng
df['year_month'] = pd.to_datetime(df['year_month'])
df['qty'] = pd.to_numeric(df['qty'], errors='coerce').fillna(0)

In [20]:
# Lọc chỉ lấy các mã hàng thuộc Nhóm 1 (ACT) để huấn luyện
df_act = df[df['fsku'].isin(nhom1_fskus)].copy()
df_act = df_act.sort_values(['fsku', 'year_month'])

In [21]:
print(df_act)

                  fsku year_month    qty
309      00.KCB.100.NS 2023-01-01   34.0
327      00.KCB.100.NS 2023-02-01   48.0
321      00.KCB.100.NS 2023-03-01   84.0
279      00.KCB.100.NS 2023-04-01   81.0
297      00.KCB.100.NS 2023-05-01  188.0
..                 ...        ...    ...
369  43.HC.ALSN5.TRANG 2025-05-01   27.0
370  43.HC.ALSN5.TRANG 2025-06-01  122.0
411  43.HC.ALSN5.TRANG 2025-07-01   94.0
393  43.HC.ALSN5.TRANG 2025-08-01   10.0
422  43.HC.ALSN5.TRANG 2025-09-01    3.0

[23891 rows x 3 columns]


In [22]:
# 2. FEATURE ENGINEERING (Xây dựng đặc trưng)
# Tạo biến trễ 12 tháng (Lag 12) để học tính mùa vụ (ví dụ: bùng nổ tháng 3 hàng năm)
df_act['lag_12'] = df_act.groupby('fsku')['qty'].shift(12)

In [23]:
# Tạo biến trung bình trượt 3 tháng (Rolling Mean) để nắm bắt xu hướng ngắn hạn
df_act['rolling_mean_3'] = df_act.groupby('fsku')['qty'].transform(
    lambda x: x.shift(1).rolling(window=3).mean()
)

In [24]:
# Loại bỏ các dòng bị rỗng (NaN) phát sinh do việc tính Lag 12 (thường là dữ liệu của năm 2023)
df_model = df_act.dropna(subset=['lag_12', 'rolling_mean_3'])

In [28]:
# 3. PHÂN CHIA TẬP TRAIN / TEST THEO THỜI GIAN
# Tập Train: Từ 01/2023 đến hết 06/2025 (đã bỏ phần NaN)
train_data = df_model[df_model['year_month'] < '2025-07-01']

In [29]:
# Tập Test: 07/2025, 08/2025, 09/2025 (3 tháng gần nhất để kiểm chứng)
test_data = df_model[df_model['year_month'] >= '2025-07-01']

In [30]:
# Định nghĩa các đặc trưng (X) và mục tiêu cần dự báo (y)
features = ['lag_12', 'rolling_mean_3']
X_train = train_data[features]
y_train = train_data['qty']
X_test = test_data[features]
y_test = test_data['qty']

In [31]:
# 4. HUẤN LUYỆN MÔ HÌNH XGBOOST
model_act = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

In [32]:
model_act.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=1000,
             n_jobs=None, num_parallel_tree=None, ...)

In [33]:
# 5. DỰ BÁO VÀ ĐÁNH GIÁ
test_predictions = model_act.predict(X_test)
mae = mean_absolute_error(y_test, test_predictions)

# Tính sai số WAPE (%) để biết mô hình lệch bao nhiêu phần trăm trên tổng sản lượng
wape = (np.sum(np.abs(y_test - test_predictions)) / np.sum(y_test)) * 100

In [34]:
print(f"Số lượng FSKU ACT được xử lý: {len(nhom1_fskus)}")

Số lượng FSKU ACT được xử lý: 1094


In [35]:
print(f"Sai số tuyệt đối trung bình (MAE): {mae:.2f} đơn vị")

Sai số tuyệt đối trung bình (MAE): 1033.66 đơn vị


In [36]:
print(f"Độ lệch toàn dụng (WAPE): {wape:.2f}%")

Độ lệch toàn dụng (WAPE): 54.09%


In [37]:
# Xuất kết quả kiểm chứng ra CSV để so sánh thực tế và dự báo
# test_data['predicted_qty'] = test_predictions
# test_data.to_csv('verification_nhom1.csv', index=False, encoding='utf-8-sig')